# Selección y extracción de características — Ejemplos

**Módulo 3 — Modelado y evaluación de enfoques no supervisados · Curso Analítica de Datos**

Este notebook acompaña las diapositivas [`3.3_Seleccion_y_Extraccion_de_Caracteristicas.pdf`](3.3_Seleccion_y_Extraccion_de_Caracteristicas.pdf) y lleva a código lo que allí se explica, usando los tres conjuntos de datos que menciona la propia presentación: **Iris**, **MNIST** y **Swiss roll**.

## Contenido

1. **Selección vs. extracción** con el dataset Iris.
2. **Criterios de selección supervisada**: Gini/Entropía, ANOVA y Chi-cuadrado.
3. **Relief** (reproduciendo el ejemplo de las frutas de las diapositivas) y **Eliminación Recursiva de Características (RFE)**.
4. **La maldición de la dimensionalidad** con un dataset de dígitos escritos a mano (estilo MNIST).
5. **Proyección a un espacio de menor dimensión** (PCA sobre un plano).
6. **Variedades y el Swiss roll**: por qué una proyección lineal no basta para "desenrollarlo".

> 💡 Este notebook **no requiere descargar nada ni tener cuenta de Kaggle**: los tres datasets (Iris, dígitos, Swiss roll) vienen incluidos directamente en `scikit-learn`. Solo necesitas tener instalado `scikit-learn`, además de `pandas`, `numpy`, `matplotlib` y `scipy`.

In [ ]:
# Librerías que usaremos en todo el notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.datasets import load_iris, load_digits, make_swiss_roll
from sklearn.feature_selection import f_classif, RFE, RFECV, VarianceThreshold
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.manifold import Isomap

pd.set_option('display.max_columns', 20)
plt.rcParams['figure.figsize'] = (7, 5)
rng = np.random.default_rng(42)  # semilla fija para que los ejemplos sean reproducibles

print('Librerías cargadas correctamente ✅')
print('scikit-learn listo para usarse')

---
## 1. Selección vs. extracción: el dataset Iris

Como en la diapositiva 3, cargamos **Iris** (150 flores de 3 especies, con 4 medidas cada una) y comparamos:

- **Selección**: quedarnos con un subconjunto de las columnas originales (por ejemplo, solo las medidas del pétalo).
- **Extracción**: construir una columna nueva que no existía antes, combinando las originales.

In [ ]:
iris = load_iris(as_frame=True)
datos = iris.frame.copy()
datos['especie'] = datos['target'].map(dict(enumerate(iris.target_names)))

print(f'{datos.shape[0]} flores, {len(iris.feature_names)} medidas originales: {iris.feature_names}')
datos.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colores = {'setosa': '#1565c0', 'versicolor': '#2e7d32', 'virginica': '#c62828'}

for especie, color in colores.items():
    sub = datos[datos['especie'] == especie]
    axes[0].scatter(sub['sepal length (cm)'], sub['sepal width (cm)'], label=especie, color=color, alpha=0.7)
    axes[1].scatter(sub['petal length (cm)'], sub['petal width (cm)'], label=especie, color=color, alpha=0.7)

axes[0].set_title('Medidas del SÉPALO')
axes[0].set_xlabel('largo (cm)'); axes[0].set_ylabel('ancho (cm)')
axes[1].set_title('Medidas del PÉTALO')
axes[1].set_xlabel('largo (cm)'); axes[1].set_ylabel('ancho (cm)')
axes[1].legend()
plt.tight_layout()
plt.show()

print('Con las medidas del pétalo, las tres especies quedan mucho más separadas entre sí')
print('que con las del sépalo: si tuviéramos que ELEGIR (seleccionar) solo dos medidas')
print('para distinguir especies, el pétalo es la opción más informativa.')

In [ ]:
# Extracción: construimos una característica NUEVA que no existía en los datos originales
datos['area_petalo'] = datos['petal length (cm)'] * datos['petal width (cm)']
datos['area_sepalo'] = datos['sepal length (cm)'] * datos['sepal width (cm)']

print('Área promedio del pétalo y del sépalo por especie:')
datos.groupby('especie')[['area_petalo', 'area_sepalo']].mean().round(2)

El área del pétalo (`largo × ancho`) es una característica **extraída**: no estaba en los datos originales, la construimos combinando dos columnas existentes. Fíjate en que separa las tres especies incluso mejor que cualquiera de las dos medidas por separado (0.37 → 5.72 → 11.30 cm² en promedio, sin solapamiento).

---
## 2. Criterios de selección supervisada

### 2.1 Gini y Entropía

Como en la diapositiva 4, medimos cuánto **reduce la incertidumbre** cada característica al usarla para separar las especies (ganancia de información), ajustando un árbol de un solo nivel de profundidad por cada medida.

In [ ]:
def entropia(etiquetas):
    """Entropía de Shannon (0 = grupo puro)."""
    _, conteos = np.unique(etiquetas, return_counts=True)
    p = conteos / len(etiquetas)
    return -np.sum(p * np.log2(p))

y = datos['target'].values
entropia_total = entropia(y)

print(f'Entropía total (antes de dividir): {entropia_total:.3f}\n')
print('Ganancia de información por medida (árbol de un solo corte):')
for medida in iris.feature_names:
    arbol = DecisionTreeClassifier(max_depth=1, criterion='entropy', random_state=0)
    arbol.fit(datos[[medida]], y)
    umbral = arbol.tree_.threshold[0]

    izquierda = y[datos[medida] <= umbral]
    derecha = y[datos[medida] > umbral]
    entropia_ponderada = (len(izquierda) * entropia(izquierda) + len(derecha) * entropia(derecha)) / len(y)
    ganancia = entropia_total - entropia_ponderada

    print(f'  {medida:22s} corte={umbral:.2f}  ganancia de información={ganancia:.3f}')

Las medidas del **pétalo** reducen la incertidumbre casi al máximo con un solo corte; las del sépalo bastante menos — confirmando con números lo que ya habíamos visto en el gráfico.

### 2.2 ANOVA

El test ANOVA contrasta si las medias de una característica numérica difieren significativamente entre las clases. `scikit-learn` lo implementa en `f_classif`: entre más grande el estadístico F (y más pequeño el valor p), más discrimina esa variable entre especies.

In [ ]:
F, valores_p = f_classif(datos[iris.feature_names], y)

print('Prueba ANOVA por medida:')
for medida, f_val, p_val in zip(iris.feature_names, F, valores_p):
    print(f'  {medida:22s} F={f_val:9.2f}   p={p_val:.2e}')

print('\nOtra vez, las medidas del pétalo tienen un estadístico F muchísimo más alto:')
print('sus medias difieren mucho más entre especies que las del sépalo.')

### 2.3 Chi-cuadrado

El chi-cuadrado se usa con **variables categóricas**: compara los conteos observados en una tabla de contingencia contra los que esperaríamos si las dos variables fueran independientes. Reproducimos el ejemplo ilustrativo de la propia diapositiva 4 (género vs. marca de café preferida).

In [ ]:
tabla_contingencia = pd.DataFrame(
    [[45, 15], [20, 40]],
    index=['Hombre', 'Mujer'],
    columns=['Marca A', 'Marca B'],
)
print('Tabla de contingencia (conteos observados):')
print(tabla_contingencia)

chi2_estadistico, valor_p, grados_libertad, esperado = stats.chi2_contingency(tabla_contingencia)
print(f'\nchi2 = {chi2_estadistico:.2f}   p = {valor_p:.4f}   grados de libertad = {grados_libertad}')

if valor_p < 0.05:
    print('\np < 0.05: hay evidencia de asociación entre género y marca preferida')
    print('(no parecen ser independientes).')
else:
    print('\np >= 0.05: no hay evidencia suficiente de asociación.')

---
## 3. Relief y Eliminación Recursiva de Características (RFE)

### 3.1 Relief: reproduciendo el ejemplo de las frutas

Antes de aplicarlo a un dataset real, reproducimos **exactamente** el ejemplo de la diapositiva 5 (5 frutas, comparando dulzura y tamaño) para confirmar con código el mismo resultado (+0.45 para dulzura, −0.05 para tamaño).

In [ ]:
frutas = pd.DataFrame({
    'clase':   ['Madura', 'Madura', 'Madura', 'Verde', 'Verde'],
    'dulzura': [0.8, 0.7, 0.9, 0.2, 0.3],
    'tamano':  [0.5, 0.4, 0.6, 0.5, 0.6],
}, index=['R', 'H1', 'H2', 'M1', 'M2'])
frutas

In [ ]:
referencia = frutas.loc['R']
hits = frutas.loc[['H1', 'H2']]     # misma clase que la referencia (madura)
misses = frutas.loc[['M1', 'M2']]   # clase distinta (verde)

print('Peso de cada característica = diferencia promedio con misses − diferencia promedio con hits\n')
for columna in ['dulzura', 'tamano']:
    diferencia_hits = (hits[columna] - referencia[columna]).abs().mean()
    diferencia_misses = (misses[columna] - referencia[columna]).abs().mean()
    peso = diferencia_misses - diferencia_hits
    print(f'  {columna}: hits={diferencia_hits:.2f}  misses={diferencia_misses:.2f}  peso={peso:+.2f}')

print('\nMismo resultado que la diapositiva: la dulzura distingue mucho mejor las clases')
print('(peso positivo alto) que el tamaño (peso ligeramente negativo, casi irrelevante).')

### 3.2 Relief sobre Iris

Ahora generalizamos la misma idea —comparar con el vecino más cercano de la misma clase (*hit*) y de la clase distinta más cercana (*miss*), promediando sobre muchas observaciones— y la aplicamos a las 4 medidas de Iris.

In [ ]:
def relief_simple(X, y, n_iteraciones=300, semilla=0):
    """Versión simplificada de Relief: promedia, sobre varias observaciones elegidas
    al azar, la diferencia con el vecino más cercano de otra clase (miss) menos la
    diferencia con el vecino más cercano de la misma clase (hit)."""
    generador = np.random.default_rng(semilla)
    X = np.asarray(X, dtype=float)
    # normalizamos cada columna a [0, 1] para que todas pesen lo mismo en la distancia
    X_norm = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0) + 1e-12)
    n, d = X.shape
    pesos = np.zeros(d)

    for _ in range(n_iteraciones):
        i = generador.integers(n)
        misma_clase = np.where((y == y[i]) & (np.arange(n) != i))[0]
        otra_clase = np.where(y != y[i])[0]

        hit = misma_clase[np.argmin(np.sum((X_norm[misma_clase] - X_norm[i]) ** 2, axis=1))]
        miss = otra_clase[np.argmin(np.sum((X_norm[otra_clase] - X_norm[i]) ** 2, axis=1))]

        pesos += np.abs(X_norm[i] - X_norm[miss]) - np.abs(X_norm[i] - X_norm[hit])

    return pesos / n_iteraciones

pesos_relief = relief_simple(datos[iris.feature_names].values, y)

print('Pesos de Relief por medida:')
for medida, peso in zip(iris.feature_names, pesos_relief):
    print(f'  {medida:22s} peso={peso:+.3f}')

print('\nRelief coincide con Gini/Entropía y ANOVA: las medidas del pétalo son, otra vez,')
print('las que mejor distinguen ejemplos de clases distintas.')

### 3.3 Eliminación Recursiva de Características (RFE)

Como en la diapositiva 5, entrenamos un modelo, retiramos la característica menos importante, y repetimos — hasta quedarnos con el número de características indicado. `RFECV` además usa validación cruzada para sugerir automáticamente cuántas conservar.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    datos[iris.feature_names], y, test_size=0.3, random_state=0, stratify=y
)

modelo = LogisticRegression(max_iter=1000)
rfe = RFE(modelo, n_features_to_select=2)
rfe.fit(X_train, y_train)  # el ranking se calcula SOLO con entrenamiento

print('Ranking de RFE (1 = más importante):')
for medida, ranking, seleccionada in zip(iris.feature_names, rfe.ranking_, rfe.support_):
    marca = '✅ seleccionada' if seleccionada else f'eliminada en el paso {ranking}'
    print(f'  {medida:22s} {marca}')

rfecv = RFECV(modelo, min_features_to_select=1, cv=5)
rfecv.fit(X_train, y_train)
print(f'\nRFECV (con validación cruzada) sugiere conservar {rfecv.n_features_} característica(s).')

---
## 4. La maldición de la dimensionalidad y el ejemplo MNIST

Como en la diapositiva 7, usamos un dataset de **dígitos escritos a mano** (imágenes de 8×8 píxeles = 64 características, el mismo espíritu que MNIST pero más liviano para experimentar sin descargar nada) para ver qué píxeles son informativos y cuáles no.

In [ ]:
digitos = load_digits()
print(f'{digitos.data.shape[0]} imágenes, {digitos.data.shape[1]} píxeles (características) cada una')

fig, axes = plt.subplots(1, 5, figsize=(10, 2.5))
for ax, imagen, etiqueta in zip(axes, digitos.images[:5], digitos.target[:5]):
    ax.imshow(imagen, cmap='gray')
    ax.set_title(f'dígito: {etiqueta}')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Varianza de cada píxel a lo largo de las ~1800 imágenes: un píxel que casi
# nunca cambia (siempre negro, por ejemplo, en las esquinas) aporta poca información.
varianza_por_pixel = digitos.data.var(axis=0).reshape(8, 8)

fig, ax = plt.subplots()
mapa = ax.imshow(varianza_por_pixel, cmap='hot')
plt.colorbar(mapa, label='Varianza')
ax.set_title('Varianza de cada píxel entre todas las imágenes')
plt.tight_layout()
plt.show()

print('Los píxeles de las esquinas y los bordes (casi negros en el mapa) varían muy poco:')
print('siempre son fondo, sin importar qué dígito se dibujó. Los píxeles centrales')
print('(en amarillo/blanco) son los que realmente distinguen un dígito de otro.')

In [ ]:
# Selección: eliminar directamente los píxeles con menos variación
seleccion_varianza = VarianceThreshold(threshold=0.5)
digitos_reducidos = seleccion_varianza.fit_transform(digitos.data)
print(f'Selección: {digitos.data.shape[1]} píxeles originales -> {digitos_reducidos.shape[1]} tras descartar los de baja varianza.')

# Extracción: en vez de descartar píxeles, construir un puñado de combinaciones (componentes) con PCA
pca_digitos = PCA(n_components=2)
digitos_2d = pca_digitos.fit_transform(digitos.data)
varianza_explicada = pca_digitos.explained_variance_ratio_.sum()
print(f'Extracción (PCA): 64 píxeles -> 2 componentes, que explican el {varianza_explicada:.1%} de la varianza total.')

fig, ax = plt.subplots()
dispersión = ax.scatter(digitos_2d[:, 0], digitos_2d[:, 1], c=digitos.target, cmap='tab10', s=10, alpha=0.7)
plt.colorbar(dispersión, label='Dígito', ticks=range(10))
ax.set_xlabel('Componente 1'); ax.set_ylabel('Componente 2')
ax.set_title('Los 64 píxeles, reducidos a solo 2 componentes con PCA')
plt.tight_layout()
plt.show()

print('Con apenas 2 de los 64 píxeles originales "resumidos" en 2 componentes, ya se distinguen')
print('agrupaciones por dígito — aunque, como advierte la diapositiva, se pierde información')
print(f'(el {1 - varianza_explicada:.1%} restante de la varianza) al comprimir tanto.')

---
## 5. Proyección a un espacio de menor dimensión

Como en la diapositiva 8, generamos puntos que viven en 3 dimensiones pero que en realidad están muy cerca de un **plano** (2 dimensiones), y usamos PCA para encontrar y proyectar sobre ese plano.

In [ ]:
n_puntos = 100
x1 = rng.uniform(-1.5, 1.5, n_puntos)
x2 = rng.uniform(-1, 1, n_puntos)
ruido = rng.normal(0, 0.05, n_puntos)  # un poquito de ruido fuera del plano
x3 = 0.3 * x1 - 0.2 * x2 + ruido        # x3 depende (casi) linealmente de x1 y x2

datos_3d = np.column_stack([x1, x2, x3])

fig = plt.figure(figsize=(11, 5))
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.scatter(x1, x2, x3, color='#1565c0', alpha=0.7)
ax1.set_xlabel('x1'); ax1.set_ylabel('x2'); ax1.set_zlabel('x3')
ax1.set_title('Datos originales en 3D')

pca_plano = PCA(n_components=2)
proyeccion_2d = pca_plano.fit_transform(datos_3d)

ax2 = fig.add_subplot(1, 2, 2)
ax2.scatter(proyeccion_2d[:, 0], proyeccion_2d[:, 1], color='#1565c0', alpha=0.7)
ax2.set_xlabel('z1'); ax2.set_ylabel('z2')
ax2.set_title('Proyección sobre el plano (PCA a 2D)')

plt.tight_layout()
plt.show()

print('Varianza explicada por cada componente:', np.round(pca_plano.explained_variance_ratio_, 3))
print(f'Entre los dos componentes explican el {pca_plano.explained_variance_ratio_.sum():.1%} de la variación total:')
print('casi toda la información de las 3 dimensiones originales cabe en el plano de 2.')

---
## 6. Variedades y el ejemplo Swiss roll

Como en la diapositiva 9, generamos el clásico **Swiss roll**: una superficie bidimensional enrollada en un espacio de 3 dimensiones. Comparamos qué pasa al reducirlo a 2D con una proyección **lineal** (PCA) frente a un método de **aprendizaje de variedades** (Isomap).

In [ ]:
X_roll, posicion_en_el_rollo = make_swiss_roll(n_samples=1500, noise=0.05, random_state=0)

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(projection='3d')
ax.scatter(X_roll[:, 0], X_roll[:, 1], X_roll[:, 2], c=posicion_en_el_rollo, cmap='rainbow', s=8)
ax.set_title('Swiss roll en su espacio original (3D)')
plt.tight_layout()
plt.show()

print('El color representa la posición real de cada punto a lo largo de la superficie enrollada')
print('(no es una cuarta variable: es la referencia que usaremos para juzgar si cada método')
print('logra "desenrollar" correctamente la superficie).')

In [ ]:
pca_rollo = PCA(n_components=2).fit_transform(X_roll)
isomap_rollo = Isomap(n_neighbors=10, n_components=2).fit_transform(X_roll)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(pca_rollo[:, 0], pca_rollo[:, 1], c=posicion_en_el_rollo, cmap='rainbow', s=8)
axes[0].set_title('Proyección LINEAL (PCA)')

axes[1].scatter(isomap_rollo[:, 0], isomap_rollo[:, 1], c=posicion_en_el_rollo, cmap='rainbow', s=8)
axes[1].set_title('Aprendizaje de VARIEDADES (Isomap)')

plt.tight_layout()
plt.show()

print('Con PCA los colores quedan mezclados: la proyección lineal aplasta el rollo y')
print('superpone capas que en realidad estaban separadas (justo la advertencia de la diapositiva 9).')
print('Con Isomap, el degradado de colores queda ordenado de un extremo a otro: el método')
print('logró "desenrollar" la superficie respetando su estructura interna.')

---
## 7. Antes de cerrar: selecciona con entrenamiento, evalúa con prueba

Como recuerda la diapositiva 10, cualquier método de selección o extracción debe **ajustarse solo con los datos de entrenamiento** y aplicarse después a prueba — el mismo principio de fuga de información que vimos en el Módulo 2.

In [ ]:
from sklearn.feature_selection import SelectKBest

# Ajustamos el selector SOLO con entrenamiento...
selector = SelectKBest(score_func=f_classif, k=2)
selector.fit(X_train, y_train)

# ...y lo aplicamos igual a entrenamiento y a prueba, sin volver a calcular nada con `test`.
X_train_reducido = selector.transform(X_train)
X_test_reducido = selector.transform(X_test)

caracteristicas_elegidas = np.array(iris.feature_names)[selector.get_support()]
print('Características elegidas (con entrenamiento únicamente):', list(caracteristicas_elegidas))
print('Forma de X_train tras la selección:', X_train_reducido.shape)
print('Forma de X_test tras la selección:', X_test_reducido.shape)

---
## 8. Para pensar y discutir en clase

Retomando las preguntas de la diapositiva 10:

1. **Iris**: según los resultados de las secciones 1 a 3 (gráficos, Gini/Entropía, ANOVA, Relief, RFE), ¿qué medidas conservarías para un modelo de clasificación? ¿Coinciden todos los criterios entre sí?
2. **Dígitos (MNIST)**: mirando el mapa de varianza de la sección 4, ¿qué píxeles eliminarías con más confianza? ¿En qué casos podría perderse información importante al eliminarlos (piensa, por ejemplo, en dígitos escritos de forma poco común)?
3. **Swiss roll**: ¿por qué PCA no logra "desenrollar" la superficie mientras que Isomap sí? ¿Qué tipo de estructura de los datos hace necesario un método de variedades en vez de una simple proyección lineal?

No hay una única respuesta "correcta": lo importante es justificar cada decisión con la evidencia obtenida en el propio notebook.

---
## Cierre

En este notebook llevamos a código las ideas de las diapositivas [`3.3_Seleccion_y_Extraccion_de_Caracteristicas.pdf`](3.3_Seleccion_y_Extraccion_de_Caracteristicas.pdf):

- La diferencia entre **seleccionar** (quedarse con columnas existentes) y **extraer** (construir columnas nuevas).
- Tres criterios de **selección supervisada**: Gini/Entropía, ANOVA y Chi-cuadrado.
- **Relief** y **RFE**, dos formas distintas de rankear o eliminar características.
- La **maldición de la dimensionalidad**, ilustrada con la varianza de los píxeles de un dataset de dígitos.
- Cómo **PCA proyecta datos sobre un subespacio de menor dimensión**.
- Por qué una superficie enrollada (**Swiss roll**) necesita **aprendizaje de variedades** (Isomap) en vez de una proyección lineal.
- La importancia de ajustar la selección/extracción **solo con entrenamiento**.

### Recursos adicionales
- [Documentación de `sklearn.feature_selection`](https://scikit-learn.org/stable/modules/feature_selection.html)
- [Documentación de `sklearn.decomposition.PCA`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)
- [Documentación de `sklearn.manifold.Isomap`](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.Isomap.html)
- [Galería de ejemplos de reducción de dimensión de scikit-learn](https://scikit-learn.org/stable/auto_examples/manifold/plot_compare_methods.html)